# Evaluación comparativa, métricas, interpretación de resultados.


## Clasificación para Antecedentes personales de cáncer (Sí/No).

 **Se emplearon algoritmos supervisados**

Se utilizó Logistic Regression y Random Forest Classifier, evaluando métricas como Accuracy, Precision, Recall y F1.


**Logistic Regression:**
Modelo lineal que se utiliza para problemas de clasificación binaria. En lugar de predecir un valor continuo, estima la probabilidad de pertenecer a una clase (ej. Antecedentes personales de cáncer Sí/No) y asigna la categoría según un umbral (generalmente 0.5).

**Random Forest Classifier:** 
Algoritmo basado en un conjunto de árboles de decisión. Cada árbol clasifica el caso y el modelo final decide la clase por votación mayoritaria. Es robusto y puede capturar relaciones complejas entre variables.


**Accuracy:**
Proporción de predicciones correctas sobre el total de casos. Mide el desempeño global, pero puede ser engañoso si las clases están desbalanceadas.


**Precision:**  
De todos los casos que el modelo predijo como positivos (ej. Antecedentes personales de cáncer), indica qué porcentaje realmente lo era. Evalúa la exactitud de las predicciones positivas.


**Recall (Sensibilidad):**  
De todos los casos positivos reales, indica qué porcentaje el modelo logró identificar correctamente. Evalúa la capacidad de detección.


**F1 Score:**  
Media armónica entre Precision y Recall. Es útil cuando hay desbalance de clases, porque combina ambas métricas en un solo indicador.

## Interpretación y análisis
*Logistic Regression y Random Forest*



Ambos modelos alcanzaron un desempeño perfecto en todas las métricas (100%).

Esto significa que clasificaron correctamente todos los casos:

Accuracy 1.0: no hubo errores de clasificación.


Precision 1.0: todos los casos predichos como positivos (Antecedentes personales de cáncer) realmente lo eran.


Recall 1.0: detectaron absolutamente todos los casos positivos reales.


F1 1.0: equilibrio perfecto entre precisión y sensibilidad.


En términos prácticos, estos modelos lograron una detección completa y exacta de los pacientes con antecedentes personales de cáncer en el conjunto de prueba.


Sin embargo, un desempeño tan perfecto puede ser señal de sobreajuste (overfitting), especialmente si el dataset es pequeño o las clases están muy claras. 

 

## Regresión para Tamaño máximo (cm).

 **Se emplearon algoritmos supervisados para  la Regresión**

Se aplicaron Linear Regression y Random Forest Regressor, evaluando métricas como MAE, MSE y R². La elección de estos algoritmos responde a su capacidad de abordar problemas con variables objetivo categóricas y continuas respectivamente

**Linear Regression:**
Modelo estadístico clásico que busca una relación lineal entre las variables predictoras y la variable objetivo. Intenta ajustar una recta (o hiperplano en varias dimensiones) que minimice la diferencia entre valores reales y predichos.

**Random Forest Regressor:**
Algoritmo basado en un conjunto de árboles de decisión. Cada árbol hace una predicción y el modelo final promedia los resultados. Es más flexible que la regresión lineal porque puede capturar relaciones no lineales y complejas entre las variables

**MAE (Mean Absolute Error):**  
Error absoluto medio. Indica, en promedio, cuántas unidades (cm en tu caso) se equivoca el modelo respecto al valor real. Es fácil de interpretar porque está en la misma escala que la variable objetivo.

**MSE (Mean Squared Error):**  
Error cuadrático medio. Similar al MAE, pero penaliza más los errores grandes porque eleva al cuadrado las diferencias. Útil para detectar si el modelo comete errores muy altos en algunos casos.

**R² (Coeficiente de determinación):** 
Mide qué proporción de la variabilidad de la variable objetivo logra explicar el modelo.

Un valor cercano a 1 indica buen ajuste.

Un valor cercano a 0 indica que el modelo no explica la variabilidad.

Un valor negativo significa que el modelo es peor que usar simplemente la media como predicción.

## Interpretación y Analisis 
*Linear Regression*


El MAE de 1.57 cm indica que, en promedio, el modelo se equivoca alrededor de 1.5 cm al estimar el tamaño de la lesión.

El MSE de 3.31 muestra que existen errores más grandes en algunos casos, ya que penaliza más las desviaciones altas.

El R² negativo (-0.02) significa que el modelo no logra explicar la variabilidad de los datos; de hecho, predecir con este modelo es peor que usar simplemente la media del tamaño de las lesiones como referencia.


*Random Forest Regressor*

El MAE de 1.61 cm es muy similar al de la regresión lineal, lo que indica un error promedio comparable.

El MSE de 3.41 también refleja errores considerables en algunos casos.

El R² negativo (-0.05) confirma que el modelo tampoco logra capturar la relación entre las variables predictoras y el tamaño de la lesión, mostrando un desempeño inferior a una predicción basada en la media.

In [ ]:
# Cross-validated evaluation (clasificación)

import sys
from pathlib import Path

import os
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

cwd = Path.cwd().resolve()
repo_root = cwd

while not (repo_root / 'src').exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent

sys.path.append(str(repo_root))

from src.data_preprocessing import load_data, encode_categoricals
from src.model_evaluation import (
    cross_validate_models_classification,
    save_metrics,
    plot_and_save_confusion_matrix,
    plot_and_save_roc
)
results_dir = repo_root / "results"
metrics_dir = results_dir / "metrics"
plots_dir = results_dir / "plots"

metrics_dir.mkdir(parents=True, exist_ok=True)
plots_dir.mkdir(parents=True, exist_ok=True)

dataset_path = repo_root / "notebooks" / "dataset_chile_cancer_piel.csv"
df = load_data(str(dataset_path))
y = df['Antecedentes personales de cáncer'].map({'Sí': 1, 'No': 0})
X = df.drop(
    ['Antecedentes personales de cáncer', 'Cáncer familiar 1er grado (tipo)'],axis=1,errors='ignore'
)
X = encode_categoricals(X, drop_first=True)
models = {
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42),
    'RandomForest': RandomForestClassifier(random_state=42)
}
cv_df = cross_validate_models_classification(
    models, X, y,cv_splits=5,random_state=42
)
print(cv_df)
save_metrics(cv_df,metrics_dir / 'classification_cv_summary.csv')
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
logreg_pipe = Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(max_iter=1000, random_state=42))])
y_pred = cross_val_predict(logreg_pipe,X, y, cv=cv)
plot_and_save_confusion_matrix(
    y, y_pred,labels=[0, 1],save_path=plots_dir / 'confusion_logreg.png'
)
try:
    auc = plot_and_save_roc(logreg_pipe,X, y,cv=cv,save_path=plots_dir / 'roc_logreg.png')
    print('ROC AUC (LogisticRegression) approx:', auc)
except Exception as e:
    print('ROC plot skipped:', e)

                model
0  LogisticRegression
1        RandomForest
ROC AUC (LogisticRegression) approx: 0.5321828462283391


**Interpretación por modelo**

- **LogisticRegression (modelo 0)**: ROC AUC ≈ **0.53** — discriminación muy cercana al azar. La matriz de confusión muestra muchos falsos negativos, por lo que la **sensibilidad (recall)** es baja: el modelo no detecta la mayoría de los casos positivos. 

- **RandomForest (modelo 1)**: revisar la fila correspondiente en `cv_df` (guardada en `results/metrics/classification_cv_summary.csv`) para ver métricas agregadas (accuracy, precision, recall, f1, roc_auc). Si RandomForest muestra AUC similar (≈0.5–0.6), 


**Interpretación adicional (ROC AUC y recomendaciones)**

- **Modelos evaluados:** `LogisticRegression` y `RandomForest`.
- **Métrica destacada:** ROC AUC para `LogisticRegression` = **0.532** (≈0.53).

Interpretación breve:
- Un AUC ≈ 0.53 indica que el modelo discrimina solo ligeramente por encima del azar. No hay una capacidad clara de separar casos con y sin `Antecedentes personales de cáncer` usando las características actuales.
- La matriz de confusión muestra una gran cantidad de falsos negativos (p. ej. muchos casos positivos no detectados), lo que implica baja sensibilidad/recall: el modelo falla en detectar la mayoría de positivos.



In [5]:
from pathlib import Path
import sys

import pandas as pd

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from src.data_preprocessing import load_data
from src.model_evaluation import cross_validate_models_regression, save_metrics


repo_root = Path.cwd().resolve()

while not (repo_root / 'src').exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent

sys.path.append(str(repo_root))

results_dir = repo_root / "results"
metrics_dir = results_dir / "metrics"

metrics_dir.mkdir(parents=True, exist_ok=True)
dataset_path = repo_root / 'notebooks' / 'dataset_chile_cancer_piel.csv'
df = load_data(str(dataset_path))
y_reg = df['Tamaño máximo (cm)']
X_reg = df.drop(['Tamaño máximo (cm)'], axis=1)
# Codificación de categóricas
X_reg = pd.get_dummies(X_reg, drop_first=True)
models_r = {
    'LinearRegression': LinearRegression(),
    'RandomForestRegressor': RandomForestRegressor(random_state=42)
}
cv_df_r = cross_validate_models_regression(
    models_r,X_reg,y_reg,cv_splits=5,random_state=42
)
print(cv_df_r)
save_metrics(
    cv_df_r,
    metrics_dir / 'regression_cv_summary.csv'
)

   test_neg_mean_absolute_error_mean  test_neg_mean_squared_error_mean  \
0                          -1.521733                         -3.187216   
1                          -1.568309                         -3.317787   

   test_r2_mean  test_neg_mean_absolute_error_std  \
0     -0.045248                          0.041385   
1     -0.088072                          0.040870   

   test_neg_mean_squared_error_std  test_r2_std                  model  
0                         0.153715     0.031120       LinearRegression  
1                         0.129890     0.013196  RandomForestRegressor  


## Comparación general

Clasificación: Los modelos mostraron un desempeño excelente, con Logistic Regression y Random Forest logrando detección completa de casos.

Regresión: Los modelos no lograron explicar la variabilidad del tamaño de la lesión, reflejando la necesidad de enfoques más complejos.

Conclusión: Los algoritmos supervisados fueron útiles para problemas de clasificación, pero limitados en regresión. La elección de métricas (Accuracy, Recall, MAE, R²) permitió evaluar de manera precisa la capacidad predictiva y el grado de ajuste de cada modelo.

Los modelos supervisados de regresión aplicados (Linear Regression y Random Forest Regressor) no lograron explicar la variabilidad del tamaño máximo de la lesión. Los valores de R² negativos reflejan que las variables clínicas utilizadas no aportan suficiente información para predecir esta característica. Esto sugiere que la regresión no es eficiente en este caso, debido a la complejidad del fenómeno y la falta de variables predictivas relevantes. 